# SupportIQ DistilBERT

Select Runtime → Change runtime type → GPU (T4 if offered). Run cells in order. Keep your frozen CSVs private. Only the optional last cell publishes model files.

In [ ]:
import sys, shutil
from pathlib import Path
assert shutil.which("uv") or not __import__("subprocess").check_call([sys.executable, "-m", "pip", "install", "uv"])
!git clone https://github.com/Sofia2200031904/supportiq-ai-platform.git /content/supportiq
%cd /content/supportiq
!uv venv --python 3.11 /content/supportiq-env
!uv pip install --python /content/supportiq-env/bin/python torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --python /content/supportiq-env/bin/python -r requirements-ml.txt 'transformers==4.57.1' 'datasets<5' 'accelerate<2' onnx onnxruntime
!/content/supportiq-env/bin/python -c "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name())"


Create a ZIP containing train.csv, val.csv and test.csv at its root from your local data/processed-2025-v2 folder. Choose that ZIP below. The training script verifies the three files against the committed hashes.

In [ ]:
import io, zipfile
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1
splits = Path("/content/frozen")
splits.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in ("train.csv", "val.csv", "test.csv"):
        (splits / name).write_bytes(archive.read(name))
del uploaded


In [ ]:
import subprocess
py = "/content/supportiq-env/bin/python"
subprocess.run([py, "-m", "ml.finetune", "--data", str(splits), "--output", "/content/distilbert"], check=True)
print(Path("/content/distilbert/results.json").read_text())


In [ ]:
subprocess.run([py, "-m", "ml.export_distilbert", "/content/distilbert/model"], check=True)
shutil.make_archive("/content/supportiq-transformer", "zip", "/content", "distilbert/model")
with zipfile.ZipFile("/content/supportiq-transformer.zip", "a", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write("/content/distilbert/results.json", "results.json")
    archive.write("/content/supportiq/tracking.db", "tracking.db")
files.download("/content/supportiq-transformer.zip")


Optional publication. Review the saved model and metrics first. Add HF_TOKEN to Colab Secrets and grant this notebook access. This uploads the tokenizer, weights and quantized ONNX, with no CSVs or training checkpoints.

In [ ]:
from google.colab import userdata
import os
env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"))
subprocess.run([py, "-m", "ml.export_distilbert", "/content/distilbert/model", "--publish"], env=env, check=True)
